# LG ThinQ (Android) 리뷰 수집 & 워드클라우드 분석 노트북

이 노트북은 두 가지 경로로 Google Play 리뷰(댓글)를 수집하여 텍스트 마이닝/워드클라우드를 생성합니다.

## 1. 공식 Google Play Developer Publishing API (리뷰 엔드포인트)
- 요구 조건: 당신(또는 조직)이 해당 앱(Package Name)의 **Play Console 퍼블리셔 권한**을 가지고 있어야 함.
- 준비 단계:
  1. GCP 프로젝트 생성 후 *Google Play Android Developer API* 활성화.
  2. 서비스 계정 생성 (JSON Key 다운로드).
  3. Play Console → 설정 → 사용자 및 권한 → 서비스 계정에 앱(또는 전체) 접근 권한 부여 (권한: "보기 금융 데이터 제외 모든 권한" 또는 리뷰 조회 가능 권한).
  4. 이 노트북에서 서비스 계정 JSON 경로와 패키지명 설정.
- 제한: **앱을 소유(권한)하지 않았다면 ThinQ(예: com.lgeha.nuts) 리뷰를 공식 API로 읽을 수 없음.**

## 2. 비공식 파이썬 라이브러리 (google-play-scraper)
- 공개 Store 페이지를 파싱. ToS 위반 가능성/변경 가능성 존재 → 연구/프로토타입 용도로만 사용 권장.
- 많은 요청/빈번 호출은 차단 위험 → sleep & batch 저장.

## 출력
- 원시 리뷰 CSV (data/thinq/thinq_reviews_raw.csv)
- 전처리/토큰/감성/워드클라우드 이미지 (data/thinq/*.png)

## 감성 처리(간단 버전)
- 별점(starRating)을 기반으로 긍/부정 라벨 (>=4 긍정, <=2 부정, 3 중립).
- 선택적으로 추가 사전 가중치 가능.

---
아래 순서대로 실행하세요. 먼저 공식 API 자격이 있는지 HAVE_PUBLISHER_ACCESS 플래그로 제어합니다.

In [1]:
# 0. 설치 & 기본 임포트 (필요 패키지 동적 설치)
import sys, subprocess, importlib, os
pkgs = ["google-api-python-client", "google-auth", "google-auth-httplib2", "google-auth-oauthlib", "google-play-scraper", "pandas", "numpy", "tqdm", "matplotlib", "seaborn", "wordcloud", "python-dotenv"]
for p in pkgs:
    try:
        importlib.import_module(p.split('==')[0].replace('-','_'))
    except ImportError:
        print('Installing', p); subprocess.check_call([sys.executable,'-m','pip','install',p])

import pandas as pd, numpy as np, re, json, time, math, itertools
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt, seaborn as sns
from wordcloud import WordCloud
from dotenv import load_dotenv
load_dotenv()
print('환경 준비 완료')

Installing google-api-python-client
Installing google-auth
Installing google-auth
Installing google-auth-oauthlib
Installing google-auth-oauthlib
Installing google-play-scraper
Installing google-play-scraper
Installing python-dotenv
Installing python-dotenv
환경 준비 완료
환경 준비 완료


In [2]:
# 1. 환경 & 설정
# LG ThinQ 추정 패키지명 (실제 Play 스토어에서 확인 필요)
PACKAGE_NAME = os.getenv('THINQ_PACKAGE', 'com.lgeha.nuts')  # 환경변수 THINQ_PACKAGE 있으면 우선
# 서비스 계정 JSON 경로 (직접 지정하거나 환경변수 GOOGLE_APPLICATION_CREDENTIALS 사용)
SERVICE_ACCOUNT_FILE = os.getenv('GOOGLE_APPLICATION_CREDENTIALS', 'service_account.json')
HAVE_PUBLISHER_ACCESS = False  # 공식 리뷰 API 접근 권한이 실제로 있을 경우 True 로 변경
MAX_OFFICIAL_PAGES = 5         # 페이지 제한 (pageToken 반복)
OFFICIAL_SLEEP = 0.5
# 비공식 수집 설정
UNOFFICIAL_TOTAL = 300         # 초기 테스트를 위해 300으로 축소 (안정 후 1200+로 확장)
UNOFFICIAL_BATCH = 200
UNOFFICIAL_SLEEP = 1.0
DATA_DIR = Path('data/thinq'); DATA_DIR.mkdir(parents=True, exist_ok=True)
RAW_FILE = DATA_DIR/'thinq_reviews_raw.csv'
print('Package:', PACKAGE_NAME)

Package: com.lgeha.nuts


### 2. 공식 Google Play Developer API 수집 함수
- 서비스 계정 JSON 키 필요
- Scope: https://www.googleapis.com/auth/androidpublisher
- reviews().list 이용 (translationLanguage='en' 예시)
- 반환 구조 예: { reviews: [ { reviewId, comments: [ { userComment: { text, starRating, lastModified: {seconds}, reviewerLanguage ... } } ] } ] }
- pageToken 으로 페이지네이션

In [4]:
# 2. 공식 API 수집 구현
from typing import List, Dict
def fetch_reviews_official(package_name: str, service_account_file: str, max_pages: int = 3, sleep: float = 0.5) -> List[Dict]:
    from google.oauth2 import service_account
    from googleapiclient.discovery import build
    if not os.path.exists(service_account_file):
        raise FileNotFoundError(f'Service account file not found: {service_account_file}')
    creds = service_account.Credentials.from_service_account_file(service_account_file, scopes=['https://www.googleapis.com/auth/androidpublisher'])
    service = build('androidpublisher','v3', credentials=creds, cache_discovery=False)
    out=[]; token=None; page=0
    while True:
        page += 1
        req = service.reviews().list(packageName=package_name, token=token, translationLanguage='en') if token else service.reviews().list(packageName=package_name, translationLanguage='en')
        resp = req.execute()
        for rv in resp.get('reviews', []):
            rid = rv.get('reviewId')
            for c in rv.get('comments', []):
                uc = c.get('userComment', {})
                out.append({
                    'review_id': rid,
                    'text': uc.get('text',''),
                    'star': uc.get('starRating'),
                    'last_modified_seconds': uc.get('lastModified',{}).get('seconds'),
                    'reviewer_language': uc.get('reviewerLanguage'),
                    'device': uc.get('device'),
                    'android_os_version': uc.get('androidOsVersion'),
                    'thumbs_up': uc.get('thumbsUpCount'),
                    'app_version_name': uc.get('appVersionName'),
                    'original_text': uc.get('originalText','')
                })
        token = resp.get('tokenPagination', {}).get('nextPageToken')
        if not token: break
        if page >= max_pages: break
        time.sleep(sleep)
    return out

official_reviews = []
if HAVE_PUBLISHER_ACCESS:
    try:
        official_reviews = fetch_reviews_official(PACKAGE_NAME, SERVICE_ACCOUNT_FILE, max_pages=MAX_OFFICIAL_PAGES, sleep=OFFICIAL_SLEEP)
        print('Official fetched:', len(official_reviews))
    except Exception as e:
        print('[Official API 실패]', e)
else:
    print('HAVE_PUBLISHER_ACCESS=False → 공식 API 건너뜀')

HAVE_PUBLISHER_ACCESS=False → 공식 API 건너뜀


### 3. 비공식 수집 (google-play-scraper 라이브러리)
- reviews 함수 반복 호출 (continuation_token)
- 많은 양 수집 시 요청 간 지연 필요

In [ ]:
# 3. 비공식 수집
from google_play_scraper import reviews, Sort
unofficial_rows = []
if UNOFFICIAL_TOTAL > 0:
    count = 0; token = None
    pbar = tqdm(total=UNOFFICIAL_TOTAL, desc='Scraping unofficial')
    while count < UNOFFICIAL_TOTAL:
        batch_size = min(UNOFFICIAL_BATCH, UNOFFICIAL_TOTAL - count)
        try:
            result, token = reviews(
                package_name=PACKAGE_NAME,
                lang='en',
                country='us',
                sort=Sort.NEWEST,
                count=batch_size,
                continuation_token=token
            )
        except Exception as e:
            print('[WARN] scraping error:', e); time.sleep(3); continue
        for r in result:
            unofficial_rows.append({
                'review_id': r.get('reviewId'),
                'text': r.get('content',''),
                'star': r.get('score'),
                'thumbs_up': r.get('thumbsUpCount'),
                'at': r.get('at'),
                'app_version': r.get('reviewCreatedVersion'),
                'reply_content': r.get('replyContent'),
                'reply_at': r.get('repliedAt')
            })
        got = len(result); count += got; pbar.update(got)
        if not token: break
        time.sleep(UNOFFICIAL_SLEEP)
    pbar.close()
print('Unofficial fetched:', len(unofficial_rows))

Scraping unofficial:   0%|          | 0/1200 [00:00<?, ?it/s]

[WARN] scraping error: reviews() got an unexpected keyword argument 'package_name'
[WARN] scraping error: reviews() got an unexpected keyword argument 'package_name'
[WARN] scraping error: reviews() got an unexpected keyword argument 'package_name'
[WARN] scraping error: reviews() got an unexpected keyword argument 'package_name'
[WARN] scraping error: reviews() got an unexpected keyword argument 'package_name'
[WARN] scraping error: reviews() got an unexpected keyword argument 'package_name'
[WARN] scraping error: reviews() got an unexpected keyword argument 'package_name'
[WARN] scraping error: reviews() got an unexpected keyword argument 'package_name'
[WARN] scraping error: reviews() got an unexpected keyword argument 'package_name'
[WARN] scraping error: reviews() got an unexpected keyword argument 'package_name'
[WARN] scraping error: reviews() got an unexpected keyword argument 'package_name'
[WARN] scraping error: reviews() got an unexpected keyword argument 'package_name'
[WAR

### 4. 병합 & 저장
- 공식/비공식 모두 수집된 경우 review_id 기준 중복 제거

In [ ]:
# 4. 병합
all_rows = []
if official_reviews: all_rows.extend(official_reviews)
if unofficial_rows: all_rows.extend(unofficial_rows)
df = pd.DataFrame(all_rows)
if df.empty:
    print('No reviews collected. 종료.')
else:
    df.drop_duplicates(subset=['review_id','text'], inplace=True)
    df.to_csv(RAW_FILE, index=False, encoding='utf-8-sig')
    print('Saved raw reviews ->', RAW_FILE, 'rows:', len(df))
df.head(3)

### 5. 전처리 & 간단 감성 라벨
- star 기반 sentiment_label (>=4: positive, <=2: negative, 나머지 neutral)
- 텍스트 정리 및 토큰화 (영/한 stopwords)

In [ ]:
# 5. 전처리
if 'text' not in df.columns or df.empty:
    print('No text column. Skip processing.')
else:
    CLEAN = re.compile(r'[^0-9A-Za-z가-힣\s]')
    MULTI = re.compile(r'\s+')
    def clean(t):
        t = str(t).lower()
        t = CLEAN.sub(' ', t)
        return MULTI.sub(' ', t).strip()
    df['clean_text'] = df['text'].map(clean)
    EN_STOP = {'the','a','an','to','for','of','on','in','and','or','is','are','be','it','its','this','that','with','from','by','at','as','was','were','have','has','had','can','will','should','could','would','not','just','so','if','but','about','into','over','than','then','when','where','how','why','what','which','who','your','you','we','they','their','our','i','my','any','some','more','most','other','such','no','nor','too','very','also','there','here','one','two','app'}
    KO_STOP = {'것','거','수','및','등','더','이','저','하다','했다','하는','으로','있다','없는','까지','이미','또','또한','하지만','그러나','그래서','그리고','거나','에서','하게','하지','하며','하면','있는','위해','됩니다'}
    STOP = EN_STOP | KO_STOP | {'lg','thinq'}
    SPLIT = re.compile(r'\s+')
    def tokenize(t):
        toks = [w for w in SPLIT.split(t) if w]
        out=[]
        for w in toks:
            if len(w) < 2: continue
            if w in STOP: continue
            if w.isdigit(): continue
            out.append(w)
        return out
    df['tokens'] = df['clean_text'].map(tokenize)
    def label(star):
        try: s = int(star)
        except: return 'neutral'
        if s >= 4: return 'positive'
        if s <= 2: return 'negative'
        return 'neutral'
    df['sentiment_label'] = df['star'].map(label)
    print(df['sentiment_label'].value_counts())
    df.to_csv(DATA_DIR/'thinq_reviews_processed.csv', index=False, encoding='utf-8-sig')
    print('Processed file saved.')
df.head(3)

### 6. 빈도 / 워드클라우드 / 감성별 워드클라우드

In [ ]:
# 6. 시각화
if 'tokens' in df.columns and not df.empty:
    from collections import Counter
    all_tokens = list(itertools.chain.from_iterable(df['tokens']))
    freq = Counter(all_tokens)
    freq_df = pd.DataFrame(freq.most_common(50), columns=['token','count'])
    plt.figure(figsize=(10,6)); sns.barplot(data=freq_df.head(25), x='count', y='token', color='steelblue'); plt.title('Top Tokens'); plt.tight_layout(); plt.savefig(DATA_DIR/'top_tokens.png', dpi=140); plt.show()
    wc_all = WordCloud(width=1200, height=600, background_color='white', colormap='viridis').generate_from_frequencies(dict(freq.most_common(200)))
    plt.figure(figsize=(14,6)); plt.imshow(wc_all); plt.axis('off'); plt.title('All Reviews WordCloud'); plt.savefig(DATA_DIR/'wordcloud_all.png', dpi=150, bbox_inches='tight'); plt.show()
    for label, cmap in [('positive','Greens'),('negative','Reds')]:
        subset = df.loc[df.sentiment_label==label, 'clean_text']
        if subset.empty: continue
        txt = ' '.join(subset)
        wc = WordCloud(width=1100, height=550, background_color='white', colormap=cmap).generate(txt)
        plt.figure(figsize=(13,6)); plt.imshow(wc); plt.axis('off'); plt.title(f'{label.title()} WordCloud')
        plt.savefig(DATA_DIR/f'wordcloud_{label}.png', dpi=150, bbox_inches='tight'); plt.show()
    freq_df.to_csv(DATA_DIR/'token_frequency_top50.csv', index=False, encoding='utf-8-sig')
else:
    print('No tokens to visualize.')

### 7. N-gram (Bigrams) 추가 분석 (선택)

In [ ]:
# 7. Bigrams
if 'tokens' in df.columns and not df.empty:
    def bigrams(toks):
        return list(zip(toks, toks[1:])) if len(toks) > 1 else []
    all_big = []
    for t in df['tokens']:
        all_big.extend(bigrams(t))
    from collections import Counter
    big_c = Counter(all_big)
    big_df = pd.DataFrame([(' '.join(k), v) for k,v in big_c.most_common(40)], columns=['bigram','count'])
    plt.figure(figsize=(10,6)); sns.barplot(data=big_df.head(25), x='count', y='bigram', color='slateblue'); plt.title('Top Bigrams'); plt.tight_layout(); plt.savefig(DATA_DIR/'top_bigrams.png', dpi=150); plt.show()
    big_df.to_csv(DATA_DIR/'bigrams_top40.csv', index=False, encoding='utf-8-sig')
else:
    print('No tokens for bigrams.')

### 8. 다음 단계 아이디어
- 언어 감지 후 한국어/영어 분리 분석
- 형태소 분석기(Okt) 추가 (konlpy)
- 더 정교한 감성 (transformer 기반 다국어 모델)
- Aspect 기반 분류 (배터리/연결성/디바이스 연동 등 ThinQ 특화)
- 주기적 스케줄링 + 변경 감지 (GitHub Actions or cron)

공식 API 권한이 없는 경우 현재 비공식 접근만 사용되므로 사내/법무 검토 후 운영환경 적용을 결정하세요.